In [1]:
import pandas as pd
import json
import xml.etree.ElementTree as ET

In [ ]:
# If running in Colab, this will mount Drive. Otherwise use local folder './'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/'
except Exception:
    DATA_ROOT = './'
print('DATA_ROOT set to', DATA_ROOT)

Mounted at /content/drive


In [ ]:
import os
# List candidate data files under DATA_ROOT for debugging
for root, dirs, files in os.walk(DATA_ROOT):
    for file in files:
        if 'Formula1' in file or 'F1' in file or 'f1' in file:
            print(os.path.join(root, file))

/content/drive/MyDrive/Formula1_2023season_raceResults.csv
/content/drive/MyDrive/f1_circuits.xml
/content/drive/MyDrive/JSON-F1-full (1).json


# Loading the files

In [ ]:
# Load CSV using DATA_ROOT fallback
csv_path = DATA_ROOT + 'Formula1_2023season_raceResults.csv'
if not os.path.exists(csv_path):
    csv_path = 'Formula1_2023season_raceResults.csv'  # try repo root
df_csv = pd.read_csv(csv_path)
df_csv.head()

,Track,Position,No,Driver,Team,Starting Grid,Laps,Time/Retired,Points,Set Fastest Lap,Fastest Lap Time
0,Bahrain,1,1,Max Verstappen,Red Bull Racing Honda RBPT,1,57,1:33:56.736,25,No,1:36.236
1,Bahrain,2,11,Sergio Perez,Red Bull Racing Honda RBPT,2,57,+11.987,18,No,1:36.344
2,Bahrain,3,14,Fernando Alonso,Aston Martin Aramco Mercedes,5,57,+38.637,15,No,1:36.156
3,Bahrain,4,55,Carlos Sainz,Ferrari,4,57,+48.052,12,No,1:37.130
4,Bahrain,5,44,Lewis Hamilton,Mercedes,7,57,+50.977,10,No,1:36.546


In [ ]:
# Load JSON dump
json_path = DATA_ROOT + 'JSON-F1-full (1).json'
if not os.path.exists(json_path):
    json_path = 'JSON-F1-full (1).json'
with open(json_path) as f:
    json_raw = json.load(f)
races = json_raw['MRData']['RaceTable']['Races']
print(f"Number of races loaded: {len(races)}")
print(f"First race: {races[0]['raceName']}")
print(f"Results in first race: {len(races[0]['Results'])}")

Number of races loaded: 29
First race: Bahrain Grand Prix
Results in first race: 20


In [ ]:
# Load XML circuits file
xml_path = DATA_ROOT + 'f1_circuits.xml'
if not os.path.exists(xml_path):
    xml_path = 'f1_circuits.xml'
tree = ET.parse(xml_path)
root = tree.getroot()
circuits = root.findall('circuit')
print(f"Number of circuits: {len(circuits)}")
for field in circuits[0]:
    print(f"  {field.tag}: {field.text}")

Number of circuits: 40
  circuitId: None
  name: Albert Park Circuit
  country: None
  city: None
  length_km: 5278
  opened: 1953
  longitude: 144.968644
  latitude: -37.849757


# Inspecting before doing anything

In [7]:
#CSV inspect
print(df_csv.shape)
print(df_csv.dtypes)
print(df_csv.isnull().sum())

(440, 11)
Track               object
Position            object
No                   int64
Driver              object
Team                object
Starting Grid        int64
Laps                 int64
Time/Retired        object
Points               int64
Set Fastest Lap     object
Fastest Lap Time    object
dtype: object
Track                0
Position             0
No                   0
Driver               0
Team                 0
Starting Grid        0
Laps                 0
Time/Retired         0
Points               0
Set Fastest Lap      0
Fastest Lap Time    13
dtype: int64


The only thing to flag: Position is object (string) when it should eventually be numeric — that's because of the NC and DQ values we spotted earlier.

In [8]:
#JSON inspect
race = races[0]
result = race['Results'][0]

print("Race-level fields:", list(race.keys()))
print("Result-level fields:", list(result.keys()))
print("Driver fields:", list(result['Driver'].keys()))
print("Constructor fields:", list(result['Constructor'].keys()))
print("FastestLap fields:", result.get('FastestLap'))

Race-level fields: ['season', 'round', 'url', 'raceName', 'Circuit', 'date', 'time', 'Results']
Result-level fields: ['number', 'position', 'positionText', 'points', 'Driver', 'Constructor', 'grid', 'laps', 'status', 'Time', 'FastestLap']
Driver fields: ['driverId', 'permanentNumber', 'code', 'url', 'givenName', 'familyName', 'dateOfBirth', 'nationality']
Constructor fields: ['constructorId', 'url', 'name', 'nationality']
FastestLap fields: {'rank': '6', 'lap': '44', 'Time': {'time': '1:36.236'}, 'AverageSpeed': {'units': 'kph', 'speed': '202.452'}}


In [9]:
#XML inspect
missing = {}

for circuit in circuits:
    for field in circuit:
        if field.tag not in missing:
            missing[field.tag] = 0
        if field.text is None or field.text.strip() == '':
            missing[field.tag] += 1

print("Total circuits:", len(circuits))
print("Missing values per field:", missing)

Total circuits: 40
Missing values per field: {'circuitId': 40, 'name': 0, 'country': 40, 'city': 40, 'length_km': 0, 'opened': 0, 'longitude': 0, 'latitude': 0}


Inspection phase done. summary of what we found out:
- CSV: Position is string not int (NC/DQ values). 13 nulls in Fastest Lap Time. Time/Retired is mixed format.
- JSON: Heavily nested. 29 races but CSV has 22 — 7 duplicates (sprint races). All fields are strings not numbers.
- XML: circuitId, country, city completely empty across all 40 rows.

# Data cleaning Phase

CSV cleaning

In [10]:
# seeing exactly what non-numeric values exist
print(df_csv[~df_csv['Position'].str.isnumeric()][['Track', 'Driver', 'Position']].to_string())

             Track           Driver Position
17         Bahrain     Esteban Ocon       NC
18         Bahrain  Charles Leclerc       NC
19         Bahrain    Oscar Piastri       NC
38    Saudi Arabia  Alexander Albon       NC
39    Saudi Arabia     Lance Stroll       NC
57       Australia   George Russell       NC
58       Australia  Alexander Albon       NC
59       Australia  Charles Leclerc       NC
78      Azerbaijan      Guanyu Zhou       NC
79      Azerbaijan    Nyck De Vries       NC
119         Monaco     Lance Stroll       NC
158         Canada   George Russell       NC
159         Canada   Logan Sargeant       NC
179        Austria  Nico Hulkenberg       NC
198  Great Britain  Kevin Magnussen       NC
199  Great Britain     Esteban Ocon       NC
218        Hungary     Esteban Ocon       NC
219        Hungary     Pierre Gasly       NC
238        Belgium     Carlos Sainz       NC
239        Belgium    Oscar Piastri       NC
257    Netherlands      Guanyu Zhou       NC
258    Net

In [11]:
# handling position column - preserving then making numeric
# create a separate column to preserve the status
df_csv['position_status'] = df_csv['Position'].apply(
    lambda x: 'NC' if x == 'NC' else ('DQ' if x == 'DQ' else 'Finished')
)

# now cast Position to numeric, NC and DQ become NaN
df_csv['Position'] = pd.to_numeric(df_csv['Position'], errors='coerce')

print(df_csv['position_status'].value_counts())
print("\nPosition nulls:", df_csv['Position'].isnull().sum())
print(df_csv[['Track', 'Driver', 'Position', 'position_status']].head(3))

position_status
Finished    388
NC           50
DQ            2
Name: count, dtype: int64

Position nulls: 52
     Track           Driver  Position position_status
0  Bahrain   Max Verstappen       1.0        Finished
1  Bahrain     Sergio Perez       2.0        Finished
2  Bahrain  Fernando Alonso       3.0        Finished


In [13]:
# inspecting fastest lap time nulls
print("Fastest Lap Time nulls:", df_csv['Fastest Lap Time'].isnull().sum())
print(df_csv[df_csv['Fastest Lap Time'].isnull()][['Track', 'Driver', 'Position', 'position_status', 'Fastest Lap Time']])

Fastest Lap Time nulls: 13
         Track           Driver  Position position_status Fastest Lap Time
59   Australia  Charles Leclerc       NaN              NC              NaN
218    Hungary     Esteban Ocon       NaN              NC              NaN
219    Hungary     Pierre Gasly       NaN              NC              NaN
239    Belgium    Oscar Piastri       NaN              NC              NaN
279      Italy     Yuki Tsunoda       NaN              NC              NaN
298  Singapore     Yuki Tsunoda       NaN              NC              NaN
299  Singapore     Lance Stroll       NaN              NC              NaN
338      Qatar   Lewis Hamilton       NaN              NC              NaN
339      Qatar     Carlos Sainz       NaN              NC              NaN
379     Mexico     Sergio Perez       NaN              NC              NaN
397     Brazil  Kevin Magnussen       NaN              NC              NaN
398     Brazil  Alexander Albon       NaN              NC              Na

In [14]:
# inspecting time/retired column
print(df_csv['Time/Retired'].unique())

['1:33:56.736' '+11.987' '+38.637' '+48.052' '+50.977' '+54.502' '+55.873'
 '+72.647' '+73.753' '+89.774' '+90.870' '+1 lap' '+2 laps' 'DNF'
 '1:21:14.894' '+5.335' '+20.728' '+25.866' '+31.065' '+35.876' '+43.162'
 '+52.832' '+54.747' '+64.826' '+67.494' '+70.588' '+76.060' '+77.478'
 '+85.021' '+86.293' '+86.445' '2:32:28.371' '+0.179' '+0.769' '+3.082'
 '+3.320' '+3.701' '+4.939' '+5.382' '+5.713' '+6.052' '+6.513' '+6.594'
 '1:32:42.436' '+2.137' '+21.217' '+22.024' '+45.491' '+46.145' '+51.617'
 '+74.240' '+80.376' '+83.862' '+86.501' '+88.623' '+89.729' '+91.332'
 '+97.794' '+100.943' '1:27:38.241' '+5.384' '+26.305' '+33.229' '+42.511'
 '+51.249' '+52.988' '+55.670' '+58.123' '+62.945' '+64.309' '+64.754'
 '+71.637' '+72.861' '+74.950' '+78.440' '+87.717' '+88.949' '1:48:51.980'
 '+27.921' '+36.990' '+39.062' '+56.284' '+61.890' '+62.362' '+63.391'
 '1:27:57.940' '+24.090' '+32.389' '+35.812' '+45.698' '+63.320' '+64.127'
 '+69.242' '+71.878' '+73.530' '+74.419' '+75.416' '1:33:

will not convert to numeric column (misleading) - winner time is not winner gap. We'll create a category column that labels which type each row is.

In [15]:
# Categorizing time/retired
def classify_time(val):
    if pd.isnull(val):
        return 'no_time'
    if val.startswith('+') and 'lap' in val:
        return 'lapped'
    if val.startswith('+'):
        return 'gap_to_winner'
    if val in ['DNF', 'DNS']:
        return 'dnf_dns'
    return 'winner_time'

df_csv['time_type'] = df_csv['Time/Retired'].apply(classify_time)
print(df_csv['time_type'].value_counts())

time_type
gap_to_winner    284
lapped            70
dnf_dns           64
winner_time       22
Name: count, dtype: int64


In [16]:
print(284 + 70 + 64 + 22)

440


Position: Cast to numeric, saved NC/DQ into position_status.

Fastest Lap Time: Left nulls as NaN, confirmed they're all NC drivers.

 Time/Retired: Labeled each value type into new time_type column

JSON

In [17]:
# flattening the JSON
rows = []

for race in races:
    for result in race['Results']:
        rows.append({
            'season':           race['season'],
            'round':            race['round'],
            'race_name':        race['raceName'],
            'race_date':        race['date'],
            'driver_id':        result['Driver']['driverId'],
            'driver_code':      result['Driver']['code'],
            'driver_name':      result['Driver']['givenName'] + ' ' + result['Driver']['familyName'],
            'nationality':      result['Driver']['nationality'],
            'dob':              result['Driver']['dateOfBirth'],
            'constructor':      result['Constructor']['name'],
            'grid':             result['grid'],
            'position':         result['position'],
            'points':           result['points'],
            'laps':             result['laps'],
            'status':           result['status'],
        })

df_json = pd.DataFrame(rows)
print(df_json.shape)
print(df_json.head(3))

(440, 15)
  season round           race_name   race_date       driver_id driver_code  \
0   2023     1  Bahrain Grand Prix  2023-03-05  max_verstappen         VER   
1   2023     1  Bahrain Grand Prix  2023-03-05           perez         PER   
2   2023     1  Bahrain Grand Prix  2023-03-05          alonso         ALO   

       driver_name nationality         dob   constructor grid position points  \
0   Max Verstappen       Dutch  1997-09-30      Red Bull    1        1     25   
1     Sergio Pérez     Mexican  1990-01-26      Red Bull    2        2     18   
2  Fernando Alonso     Spanish  1981-07-29  Aston Martin    5        3     15   

  laps    status  
0   57  Finished  
1   57  Finished  
2   57  Finished  


In [18]:
print("Unique races:", df_json['race_name'].nunique())
print(df_json.groupby('race_name').size().unique())

Unique races: 22
[20]


In [19]:
# inspecting JSON types
print(df_json.dtypes)
print("\nNulls:", df_json.isnull().sum())

season         object
round          object
race_name      object
race_date      object
driver_id      object
driver_code    object
driver_name    object
nationality    object
dob            object
constructor    object
grid           object
position       object
points         object
laps           object
status         object
dtype: object

Nulls: season         0
round          0
race_name      0
race_date      0
driver_id      0
driver_code    0
driver_name    0
nationality    0
dob            0
constructor    0
grid           0
position       0
points         0
laps           0
status         0
dtype: int64


In [20]:
# casting each column to its correct type
df_json['round']    = pd.to_numeric(df_json['round'])
df_json['grid']     = pd.to_numeric(df_json['grid'])
df_json['position'] = pd.to_numeric(df_json['position'], errors='coerce')
df_json['points']   = pd.to_numeric(df_json['points'])
df_json['laps']     = pd.to_numeric(df_json['laps'])
df_json['race_date'] = pd.to_datetime(df_json['race_date'])
df_json['dob']       = pd.to_datetime(df_json['dob'])

print(df_json.dtypes)
print("\nPosition nulls after cast:", df_json['position'].isnull().sum())

season                 object
round                   int64
race_name              object
race_date      datetime64[ns]
driver_id              object
driver_code            object
driver_name            object
nationality            object
dob            datetime64[ns]
constructor            object
grid                    int64
position                int64
points                  int64
laps                    int64
status                 object
dtype: object

Position nulls after cast: 0


XML

In [21]:
# converting XML to dataframe
rows = []

for circuit in circuits:
    rows.append({
        'circuit_name': circuit.find('name').text,
        'length_km':    circuit.find('length_km').text,
        'opened':       circuit.find('opened').text,
        'longitude':    circuit.find('longitude').text,
        'latitude':     circuit.find('latitude').text,
    })

df_xml = pd.DataFrame(rows)
print(df_xml.shape)
print(df_xml.head(3))

(40, 5)
                     circuit_name length_km opened   longitude    latitude
0             Albert Park Circuit      5278   1953  144.968644  -37.849757
1   Bahrain International Circuit      5412   2002   50.510539   26.031766
2  Shanghai International Circuit      5451   2004   121.21838   31.336835


In [22]:
# fixing XML types
df_xml['length_km'] = pd.to_numeric(df_xml['length_km'])
df_xml['opened']    = pd.to_numeric(df_xml['opened'])
df_xml['longitude'] = pd.to_numeric(df_xml['longitude'])
df_xml['latitude']  = pd.to_numeric(df_xml['latitude'])

print(df_xml.dtypes)
print(df_xml.head(3))

circuit_name     object
length_km         int64
opened            int64
longitude       float64
latitude        float64
dtype: object
                     circuit_name  length_km  opened   longitude   latitude
0             Albert Park Circuit       5278    1953  144.968644 -37.849757
1   Bahrain International Circuit       5412    2002   50.510539  26.031766
2  Shanghai International Circuit       5451    2004  121.218380  31.336835


Cleaning done

## Join keys

In [23]:
# checking join keys
print("CSV tracks:")
print(sorted(df_csv['Track'].unique()))

print("\nJSON race names:")
print(sorted(df_json['race_name'].unique()))

print("\nXML circuit names:")
print(sorted(df_xml['circuit_name'].unique()))

CSV tracks:
['Abu Dhabi', 'Australia', 'Austria', 'Azerbaijan', 'Bahrain', 'Belgium', 'Brazil', 'Canada', 'Great Britain', 'Hungary', 'Italy', 'Japan', 'Las Vegas', 'Mexico', 'Miami', 'Monaco', 'Netherlands', 'Qatar', 'Saudi Arabia', 'Singapore', 'Spain', 'United States']

JSON race names:
['Abu Dhabi Grand Prix', 'Australian Grand Prix', 'Austrian Grand Prix', 'Azerbaijan Grand Prix', 'Bahrain Grand Prix', 'Belgian Grand Prix', 'British Grand Prix', 'Canadian Grand Prix', 'Dutch Grand Prix', 'Hungarian Grand Prix', 'Italian Grand Prix', 'Japanese Grand Prix', 'Las Vegas Grand Prix', 'Mexico City Grand Prix', 'Miami Grand Prix', 'Monaco Grand Prix', 'Qatar Grand Prix', 'Saudi Arabian Grand Prix', 'Singapore Grand Prix', 'Spanish Grand Prix', 'São Paulo Grand Prix', 'United States Grand Prix']

XML circuit names:
['Albert Park Circuit', 'Autodromo Enzo e Dino Ferrari', 'Autodromo Internazionale del Mugello', 'Autodromo Nazionale Monza', 'Autódromo Hermanos Rodríguez', 'Autódromo Interna

In [24]:
# creating join key mappings
csv_to_json = {
    'Abu Dhabi':     'Abu Dhabi Grand Prix',
    'Australia':     'Australian Grand Prix',
    'Austria':       'Austrian Grand Prix',
    'Azerbaijan':    'Azerbaijan Grand Prix',
    'Bahrain':       'Bahrain Grand Prix',
    'Belgium':       'Belgian Grand Prix',
    'Brazil':        'São Paulo Grand Prix',
    'Canada':        'Canadian Grand Prix',
    'Great Britain': 'British Grand Prix',
    'Hungary':       'Hungarian Grand Prix',
    'Italy':         'Italian Grand Prix',
    'Japan':         'Japanese Grand Prix',
    'Las Vegas':     'Las Vegas Grand Prix',
    'Mexico':        'Mexico City Grand Prix',
    'Miami':         'Miami Grand Prix',
    'Monaco':        'Monaco Grand Prix',
    'Netherlands':   'Dutch Grand Prix',
    'Qatar':         'Qatar Grand Prix',
    'Saudi Arabia':  'Saudi Arabian Grand Prix',
    'Singapore':     'Singapore Grand Prix',
    'Spain':         'Spanish Grand Prix',
    'United States': 'United States Grand Prix',
}

csv_to_xml = {
    'Abu Dhabi':     'Yas Marina Circuit',
    'Australia':     'Albert Park Circuit',
    'Austria':       'Red Bull Ring',
    'Azerbaijan':    'Baku City Circuit',
    'Bahrain':       'Bahrain International Circuit',
    'Belgium':       'Circuit de Spa-Francorchamps',
    'Brazil':        'Autódromo José Carlos Pace - Interlagos',
    'Canada':        'Circuit Gilles-Villeneuve',
    'Great Britain': 'Silverstone Circuit',
    'Hungary':       'Hungaroring',
    'Italy':         'Autodromo Nazionale Monza',
    'Japan':         'Suzuka International Racing Course',
    'Las Vegas':     'Las Vegas Street Circuit',
    'Mexico':        'Autódromo Hermanos Rodríguez',
    'Miami':         'Miami International Autodrome',
    'Monaco':        'Circuit de Monaco',
    'Netherlands':   'Circuit Zandvoort',
    'Qatar':         'Losail International Circuit',
    'Saudi Arabia':  'Jeddah Corniche Circuit',
    'Singapore':     'Marina Bay Street Circuit',
    'Spain':         'Circuit de Barcelona-Catalunya',
    'United States': 'Circuit of the Americas',
}

# adding normalized keys to csv
df_csv['race_name']    = df_csv['Track'].map(csv_to_json)
df_csv['circuit_name'] = df_csv['Track'].map(csv_to_xml)

print("Unmapped race_names:", df_csv['race_name'].isnull().sum())
print("Unmapped circuit_names:", df_csv['circuit_name'].isnull().sum())

Unmapped race_names: 0
Unmapped circuit_names: 0


# Building df_master

In [26]:
print("JSON columns:", df_json.columns.tolist())
print("CSV columns:", df_csv.columns.tolist())

JSON columns: ['season', 'round', 'race_name', 'race_date', 'driver_id', 'driver_code', 'driver_name', 'nationality', 'dob', 'constructor', 'grid', 'position', 'points', 'laps', 'status']
CSV columns: ['Track', 'Position', 'No', 'Driver', 'Team', 'Starting Grid', 'Laps', 'Time/Retired', 'Points', 'Set Fastest Lap', 'Fastest Lap Time', 'position_status', 'time_type', 'race_name', 'circuit_name']


In [27]:
csv_drivers = set(df_csv['Driver'].unique())
json_drivers = set(df_json['driver_name'].unique())

print("In CSV but not JSON:", csv_drivers - json_drivers)
print("In JSON but not CSV:", json_drivers - csv_drivers)

In CSV but not JSON: {'Nyck De Vries', 'Nico Hulkenberg', 'Sergio Perez'}
In JSON but not CSV: {'Nyck de Vries', 'Nico Hülkenberg', 'Sergio Pérez'}


In [28]:
# fixing driver name mismatches
name_fix = {
    'Sergio Pérez':   'Sergio Perez',
    'Nico Hülkenberg': 'Nico Hulkenberg',
    'Nyck de Vries':  'Nyck De Vries',
}

df_json['driver_name'] = df_json['driver_name'].replace(name_fix)

csv_drivers = set(df_csv['Driver'].unique())
json_drivers = set(df_json['driver_name'].unique())
print("Remaining mismatches:", csv_drivers - json_drivers)

Remaining mismatches: set()


In [31]:
df_master = df_csv.merge(
    df_json[['race_name', 'driver_name', 'nationality', 'dob', 'driver_code', 'driver_id', 'status']],
    left_on=['race_name', 'Driver'],
    right_on=['race_name', 'driver_name'],
    how='left'
).merge(
    df_xml,
    on='circuit_name',
    how='left'
)

print(df_master.shape)
print(df_master.columns.tolist())
print(df_master.isnull().sum())

(440, 25)
['Track', 'Position', 'No', 'Driver', 'Team', 'Starting Grid', 'Laps', 'Time/Retired', 'Points', 'Set Fastest Lap', 'Fastest Lap Time', 'position_status', 'time_type', 'race_name', 'circuit_name', 'driver_name', 'nationality', 'dob', 'driver_code', 'driver_id', 'status', 'length_km', 'opened', 'longitude', 'latitude']
Track                0
Position            52
No                   0
Driver               0
Team                 0
Starting Grid        0
Laps                 0
Time/Retired         0
Points               0
Set Fastest Lap      0
Fastest Lap Time    13
position_status      0
time_type            0
race_name            0
circuit_name         0
driver_name          0
nationality          0
dob                  0
driver_code          0
driver_id            0
status               0
length_km            0
opened               0
longitude            0
latitude             0
dtype: int64


In [32]:
# dropping duplicate column
df_master = df_master.drop(columns=['driver_name'])

print(df_master.shape)
print(df_master.columns.tolist())

(440, 24)
['Track', 'Position', 'No', 'Driver', 'Team', 'Starting Grid', 'Laps', 'Time/Retired', 'Points', 'Set Fastest Lap', 'Fastest Lap Time', 'position_status', 'time_type', 'race_name', 'circuit_name', 'nationality', 'dob', 'driver_code', 'driver_id', 'status', 'length_km', 'opened', 'longitude', 'latitude']


# Integration cleaning

In [33]:
# renaming all column to snake_case
df_master = df_master.rename(columns={
    'Track':          'track',
    'Position':       'finish_position',
    'No':             'driver_number',
    'Driver':         'driver_name',
    'Team':           'team',
    'Starting Grid':  'grid_position',
    'Laps':           'laps',
    'Time/Retired':   'time_retired',
    'Points':         'points',
    'Set Fastest Lap': 'set_fastest_lap',
    'Fastest Lap Time': 'fastest_lap_time',
})

print(df_master.columns.tolist())

['track', 'finish_position', 'driver_number', 'driver_name', 'team', 'grid_position', 'laps', 'time_retired', 'points', 'set_fastest_lap', 'fastest_lap_time', 'position_status', 'time_type', 'race_name', 'circuit_name', 'nationality', 'dob', 'driver_code', 'driver_id', 'status', 'length_km', 'opened', 'longitude', 'latitude']


In [34]:
# adding race date from json
race_date_map = df_json[['race_name', 'race_date']].drop_duplicates()

df_master = df_master.merge(
    race_date_map,
    on='race_name',
    how='left'
)

print(df_master.shape)
print(df_master['race_date'].head(5))
print(df_master['race_date'].isnull().sum())

(440, 25)
0   2023-03-05
1   2023-03-05
2   2023-03-05
3   2023-03-05
4   2023-03-05
Name: race_date, dtype: datetime64[ns]
0


In [35]:
# dropping redundant columns
df_master = df_master.drop(columns=['race_name', 'circuit_name'])

print(df_master.shape)
print(df_master.columns.tolist())

(440, 23)
['track', 'finish_position', 'driver_number', 'driver_name', 'team', 'grid_position', 'laps', 'time_retired', 'points', 'set_fastest_lap', 'fastest_lap_time', 'position_status', 'time_type', 'nationality', 'dob', 'driver_code', 'driver_id', 'status', 'length_km', 'opened', 'longitude', 'latitude', 'race_date']


In [36]:
# reordering columns
df_master = df_master[[
    # race info
    'race_date',
    'track',
    # driver info
    'driver_name',
    'driver_code',
    'driver_id',
    'driver_number',
    'nationality',
    'dob',
    'team',
    # result info
    'grid_position',
    'finish_position',
    'position_status',
    'laps',
    'time_retired',
    'time_type',
    'points',
    'set_fastest_lap',
    'fastest_lap_time',
    'status',
    # circuit info
    'length_km',
    'opened',
    'longitude',
    'latitude',
]]

print(df_master.shape)
print(df_master.head(3))

(440, 23)
   race_date    track      driver_name driver_code       driver_id  \
0 2023-03-05  Bahrain   Max Verstappen         VER  max_verstappen   
1 2023-03-05  Bahrain     Sergio Perez         PER           perez   
2 2023-03-05  Bahrain  Fernando Alonso         ALO          alonso   

   driver_number nationality        dob                          team  \
0              1       Dutch 1997-09-30    Red Bull Racing Honda RBPT   
1             11     Mexican 1990-01-26    Red Bull Racing Honda RBPT   
2             14     Spanish 1981-07-29  Aston Martin Aramco Mercedes   

   grid_position  ...  time_retired      time_type  points set_fastest_lap  \
0              1  ...   1:33:56.736    winner_time      25              No   
1              2  ...       +11.987  gap_to_winner      18              No   
2              5  ...       +38.637  gap_to_winner      15              No   

  fastest_lap_time    status length_km opened  longitude   latitude  
0         1:36.236  Finished     

# SAVING

In [37]:
df_master.to_csv('/content/drive/MyDrive/df_master.csv', index=False)
print("Saved successfully.")
print("Shape:", df_master.shape)
print("Columns:", df_master.columns.tolist())

Saved successfully.
Shape: (440, 23)
Columns: ['race_date', 'track', 'driver_name', 'driver_code', 'driver_id', 'driver_number', 'nationality', 'dob', 'team', 'grid_position', 'finish_position', 'position_status', 'laps', 'time_retired', 'time_type', 'points', 'set_fastest_lap', 'fastest_lap_time', 'status', 'length_km', 'opened', 'longitude', 'latitude']
